# GEPA Skill Optimization Pipeline

This notebook demonstrates how to optimize Claude Agent Skills using **GEPA's `optimize_anything`** API with **OpenAI GPT** models.

## Overview

1. **Setup** - Import dependencies and configure API key
2. **Load Skill** - Parse skill directory
3. **Analyze** - Check against best practices
4. **Define Evaluator** - Scoring function for GEPA
5. **Configure** - Set objective, background, and GEPA config
6. **Run GEPA** - `optimize_anything` evolves the skill content
7. **Results** - Extract optimized skill, calculate metrics
8. **Compare** - Before/after comparison

## 1. Setup

In [ ]:
# Install dependencies (run once)
# !pip install -e .
# !pip install gepa openai python-dotenv

In [ ]:
import json
import os
import re
from pathlib import Path

import gepa.optimize_anything as oa
from gepa.optimize_anything import optimize_anything, GEPAConfig, EngineConfig, ReflectionConfig
from dotenv import load_dotenv

from skillopt import SkillParser, SkillAnalyzer

load_dotenv()

parser = SkillParser()
analyzer = SkillAnalyzer()

print("SkillOpt components initialized")

In [ ]:
# Verify API key is set
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY environment variable not set")

MODEL = "openai/gpt-4o"
print(f"Using model: {MODEL}")
print("API key: set")

## 2. Load Skill

In [3]:
# Path to your skill directory
SKILL_PATH = Path("examples/Bad_Kubernetes_Helper_Skill")

# Parse the skill
if SKILL_PATH.exists():
    skill = parser.parse_directory(SKILL_PATH)
    print(f"Loaded skill: {skill.name}")
    print(f"  Lines: {skill.total_lines}")
    print(f"  References: {len(skill.references)}")
else:
    print(f"Skill not found at: {SKILL_PATH}")
    skill = None

Loaded skill: Bad_Kubernetes_Helper_Skill
  Lines: 565
  References: 2


## 3. Analyze Skill

In [4]:
# Analyze skill against best practices
if skill:
    report = analyzer.analyze(skill)
    
    print(f"Analysis Report: {skill.name}")
    print("="*50)
    print(f"Score: {report.score}/100")
    print(f"Issues: {len(report.issues)}")
    print(f"")
    print("Issues by severity:")
    errors = [i for i in report.issues if i.severity == 'error']
    warnings = [i for i in report.issues if i.severity == 'warning']
    suggestions = [i for i in report.issues if i.severity == 'suggestion']
    print(f"  Errors: {len(errors)}")
    print(f"  Warnings: {len(warnings)}")
    print(f"  Suggestions: {len(suggestions)}")

Analysis Report: Bad_Kubernetes_Helper_Skill
Score: 0/100
Issues: 22

Issues by severity:
  Errors: 2
  Warnings: 16
  Suggestions: 4


In [5]:
# View detailed issues
if skill and 'report' in dir():
    print("Detailed Issues:")
    print("="*50)
    for issue in report.issues[:-1]:
        icon = {"error": "[ERR]", "warning": "[WRN]", "suggestion": "[SUG]"}.get(issue.severity, "[?]")
        print(f"{icon} {issue.category}: {issue.message}")

Detailed Issues:
[ERR] structure: SKILL.md has 502 lines, should be under 500
[WRN] conciseness: Filler phrase 'it is important to' appears 14 time(s)
[WRN] conciseness: Filler phrase 'please note that' appears 11 time(s)
[WRN] conciseness: Filler phrase 'keep in mind' appears 7 time(s)
[WRN] conciseness: Filler phrase 'make sure to' appears 40 time(s)
[WRN] conciseness: Filler phrase 'remember to' appears 9 time(s)
[WRN] conciseness: Filler phrase 'ensure that' appears 40 time(s)
[WRN] conciseness: Filler phrase 'you should' appears 12 time(s)
[WRN] conciseness: Filler phrase 'you can' appears 6 time(s)
[WRN] conciseness: Filler phrase 'you'll need to' appears 1 time(s)
[WRN] conciseness: Filler phrase 'first, you'll' appears 1 time(s)
[WRN] conciseness: Filler phrase 'don't forget to' appears 12 time(s)
[WRN] conciseness: Filler phrase 'you need to' appears 3 time(s)
[WRN] conciseness: Assumes Claude doesn't know Kubernetes
[WRN] structure: Reference advanced-topics.md links to other

## 4. Define Evaluator

The evaluator scores each candidate skill that GEPA proposes. It returns `(score, feedback_dict)` — the feedback tells GEPA's reflection LM exactly what to improve next.

In [ ]:
# Helper functions
def count_code_blocks(content: str) -> int:
    """Count number of code blocks."""
    return len(re.findall(r'```', content)) // 2

# Filler phrases and verbose patterns to detect
FILLER_PHRASES = [
    'make sure to', 'ensure that', "don't forget to", 'remember to',
    'it is important to', 'please note that', 'keep in mind',
    'you should', 'you need to', 'you must',
]

VERBOSE_PATTERNS = [
    'Portable Document Format', 'JavaScript Object Notation',
    'Yet Another Markup Language', 'Application Programming Interface',
    'is an open-source', 'is a system for',
]

def create_skill_evaluator(original_content: str):
    """Create an evaluator closure that scores optimized skill candidates.

    Scoring weights:
    - 25% Filler phrase removal
    - 20% Conciseness (target 40-70% reduction)
    - 25% Code block preservation
    - 15% Structure preserved (frontmatter, sections)
    - 15% Verbose explanations removed
    """
    original_len = len(original_content)
    original_blocks = count_code_blocks(original_content)

    def evaluate(candidate: str) -> tuple[float, dict]:
        score = 0.0
        feedback = {}

        # 1. Filler phrase removal (25%)
        found_fillers = [p for p in FILLER_PHRASES if p.lower() in candidate.lower()]
        filler_count = len(found_fillers)
        filler_score = max(0, 1 - (filler_count / 5))
        score += 0.25 * filler_score
        feedback["filler_phrases"] = (
            f"Found {filler_count}: {found_fillers}" if found_fillers else "None found"
        )

        # 2. Conciseness — target 40-70% reduction (20%)
        optimized_len = len(candidate)
        if optimized_len < original_len:
            reduction_ratio = 1 - (optimized_len / original_len)
            if reduction_ratio < 0.4:
                concise_score = reduction_ratio / 0.4
            elif reduction_ratio <= 0.7:
                concise_score = 1.0
            else:
                concise_score = max(0, 1 - (reduction_ratio - 0.7) * 3)
        else:
            reduction_ratio = 0.0
            concise_score = 0.0
        score += 0.20 * concise_score
        feedback["size_reduction"] = f"{reduction_ratio * 100:.1f}% (target: 40-70%)"

        # 3. Code block preservation (25%)
        opt_blocks = count_code_blocks(candidate)
        if original_blocks > 0:
            block_ratio = opt_blocks / original_blocks
            if block_ratio >= 0.5:
                code_score = min(1.0, block_ratio)
            else:
                code_score = block_ratio * 0.5
        else:
            code_score = 1.0 if opt_blocks == 0 else 0.8
        score += 0.25 * code_score
        feedback["code_blocks"] = f"preserved {opt_blocks}/{original_blocks}"

        # 4. Structure preserved (15%)
        has_frontmatter = candidate.strip().startswith('---')
        has_sections = '## ' in candidate
        has_code = '```' in candidate
        structure_score = (
            (0.4 if has_frontmatter else 0) +
            (0.3 if has_sections else 0) +
            (0.3 if has_code else 0)
        )
        score += 0.15 * structure_score
        feedback["structure"] = (
            f"frontmatter={'yes' if has_frontmatter else 'MISSING'}, "
            f"sections={'yes' if has_sections else 'MISSING'}, "
            f"code={'yes' if has_code else 'MISSING'}"
        )

        # 5. Verbose explanations removed (15%)
        found_verbose = [p for p in VERBOSE_PATTERNS if p.lower() in candidate.lower()]
        verbose_count = len(found_verbose)
        verbose_score = max(0, 1 - (verbose_count / 3))
        score += 0.15 * verbose_score
        feedback["verbose_explanations"] = (
            f"Found {verbose_count}: {found_verbose}" if found_verbose else "None found"
        )

        oa.log(f"Score: {score:.3f} | Size: {feedback['size_reduction']} | "
               f"Filler: {filler_count} | Code: {feedback['code_blocks']} | "
               f"Verbose: {verbose_count}")

        return score, feedback

    return evaluate

print("Evaluator factory defined")
print("  - 25% Filler removal")
print("  - 20% Conciseness (40-70% target)")
print("  - 25% Code block preservation")
print("  - 15% Structure (frontmatter, sections)")
print("  - 15% Verbose removal")

## 5. Configure Optimization

Set up the objective, background context, and evaluator for `optimize_anything`.

In [ ]:
# Prepare skill content and issues
skill_content = skill.main_file.raw_content[:8000]
original_code_blocks = count_code_blocks(skill_content)

issues_str = "\n".join([
    f"- [{issue.severity}] {issue.category}: {issue.message}"
    for issue in report.issues[:20]
])

print(f"Skill content: {len(skill_content):,} chars")
print(f"Code blocks found: {original_code_blocks}")
print()

# Build objective and background for GEPA reflection
objective = (
    "Optimize this Claude Agent Skill (SKILL.md) to reduce token usage while "
    "preserving full functionality. Remove filler phrases, verbose explanations, "
    "and redundant content. Preserve all code blocks, YAML frontmatter, and "
    "section headers. Consolidate similar commands using placeholders. "
    "The output must be a complete, valid SKILL.md file."
)

background = (
    f"Issues found by the skill analyzer:\n{issues_str}\n\n"
    "Best practices for Claude Agent Skills:\n"
    "- Remove filler phrases: 'make sure to', 'ensure that', 'don't forget to', "
    "'remember to', 'it is important to', 'please note that', 'keep in mind'\n"
    "- Don't explain concepts Claude already knows (YAML, JSON, APIs, Kubernetes, etc.)\n"
    "- Keep YAML frontmatter (--- name/description ---), section headers (##), "
    "and all code blocks (```)\n"
    "- Consolidate similar commands using placeholders (e.g., -n <namespace>)\n"
    "- Target 40-70% size reduction — not more, not less\n"
    "- Preserve core functionality and workflow logic"
)

# Create evaluator
evaluator = create_skill_evaluator(skill_content)

print("Objective set")
print(f"Background: {len(background)} chars (issues + best practices)")
print("Evaluator created")

## 6. Run GEPA Optimization

`optimize_anything` takes the skill content as the seed candidate and evolves it via reflection-driven search, guided by the evaluator's feedback.

In [ ]:
# Run GEPA optimize_anything
MAX_METRIC_CALLS = 10

print(f"Running GEPA optimize_anything (max_metric_calls={MAX_METRIC_CALLS})...")
print("=" * 50)

result = optimize_anything(
    seed_candidate=skill_content,
    evaluator=evaluator,
    objective=objective,
    background=background,
    config=GEPAConfig(
        engine=EngineConfig(
            max_metric_calls=MAX_METRIC_CALLS,
            display_progress_bar=True,
        ),
        reflection=ReflectionConfig(
            reflection_lm=MODEL,
            reflection_lm_kwargs={"temperature": 1.0},
            skip_perfect_score=True,
        ),
    ),
)

print("\nGEPA optimization complete")
print(f"  Best score: {result.val_aggregate_scores[result.best_idx]:.3f}")
print(f"  Candidates explored: {len(result.candidates)}")
print(f"  Total metric calls: {result.total_metric_calls}")

## 7. Results

In [ ]:
# Extract optimized content
optimized_content = result.best_candidate
if isinstance(optimized_content, dict):
    optimized_content = list(optimized_content.values())[0]

# Calculate metrics
original_len = len(skill_content)
optimized_len = len(optimized_content)
reduction = original_len - optimized_len
reduction_pct = (reduction / original_len) * 100 if original_len > 0 else 0

orig_blocks = count_code_blocks(skill_content)
opt_blocks = count_code_blocks(optimized_content)

# Final evaluator score
metric_score, feedback = evaluator(optimized_content)

print(f"Results:")
print(f"  Original: {original_len:,} chars, {orig_blocks} code blocks")
print(f"  Optimized: {optimized_len:,} chars, {opt_blocks} code blocks")
print(f"  Reduction: {reduction:,} chars ({reduction_pct:.1f}%)")
print(f"  Code blocks preserved: {opt_blocks}/{orig_blocks}")
print(f"  Metric score: {metric_score:.2f}")
print()
print("Evaluator feedback:")
for k, v in feedback.items():
    print(f"  {k}: {v}")

In [15]:
# Preview the optimized content
if 'optimized_content' in dir():
    print("GEPA-Optimized Skill Preview:")
    print("="*50)
    print(optimized_content[:2000])
    if len(optimized_content) > 2000:
        print("\n... (truncated)")

GEPA-Optimized Skill Preview:
---
name: Kubernetes_Helper_Skill_For_Managing_Cluster_Resources
description: Assist with managing resources, deployments, scaling, and troubleshooting in Kubernetes clusters.
---

# Kubernetes Helper Skill

## What This Skill Does

This skill assists with Kubernetes-related tasks, including managing cluster resources, deploying applications, scaling workloads, and troubleshooting issues.

## Prerequisites and Requirements

- **kubectl**: Ensure it is installed and correctly configured.
- **Kubernetes Cluster Access**: Confirm kubeconfig is properly set up and permissions are sufficient.

## How To Use This Skill

### Step 1: Connect to Your Cluster

Configure and verify kubectl context.

```bash
kubectl config get-contexts
kubectl config use-context <your-cluster>
kubectl cluster-info
kubectl get nodes
```

### Step 2: Check the Current State

Evaluate the existing state of cluster resources.

```bash
kubectl get pods [-n <namespace>] [--all-namespaces] [

In [16]:
# Save the optimized skill
if 'optimized_content' in dir():
    OUTPUT_DIR = Path("examples/Bad_Kubernetes_Helper_Skill_GEPA_Optimized_1")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    (OUTPUT_DIR / "SKILL.md").write_text(optimized_content)
    print(f"Saved GEPA-optimized skill to: {OUTPUT_DIR / 'SKILL.md'}")

Saved GEPA-optimized skill to: examples/Bad_Kubernetes_Helper_Skill_GEPA_Optimized_1/SKILL.md


## 8. Compare Results

In [ ]:
# Compare original vs optimized
original = skill_content

print("Comparison: Original vs GEPA-Optimized")
print("=" * 60)
print(f"{'Metric':<30} {'Original':<15} {'Optimized':<15}")
print("-" * 60)
print(f"{'Characters':<30} {len(original):<15,} {len(optimized_content):<15,}")
print(f"{'Lines':<30} {original.count(chr(10)):<15} {optimized_content.count(chr(10)):<15}")
print(f"{'Est. Tokens':<30} {int(len(original)*0.25):<15,} {int(len(optimized_content)*0.25):<15,}")

orig_blocks = count_code_blocks(original)
opt_blocks = count_code_blocks(optimized_content)
print(f"{'Code Blocks':<30} {orig_blocks:<15} {opt_blocks:<15}")

filler_phrases = ['make sure to', 'ensure that', "don't forget to", 'remember to', 'it is important to']
orig_fillers = sum(original.lower().count(p) for p in filler_phrases)
opt_fillers = sum(optimized_content.lower().count(p) for p in filler_phrases)
print(f"{'Filler Phrases':<30} {orig_fillers:<15} {opt_fillers:<15}")

reduction_pct = (1 - len(optimized_content) / len(original)) * 100
print(f"\n{'Reduction':<30} {len(original) - len(optimized_content):,} chars ({reduction_pct:.1f}%)")
if orig_blocks > 0:
    print(f"{'Code Block Retention':<30} {opt_blocks}/{orig_blocks} ({100 * opt_blocks / orig_blocks:.0f}%)")

## Summary

This notebook optimizes Claude Agent Skills using **GEPA's `optimize_anything`** API.

The skill content is passed directly as the seed candidate. GEPA proposes mutations, the evaluator scores them and returns structured feedback (what filler was found, how much was reduced, which code blocks survived), and GEPA's reflection LM uses that feedback to guide the next proposal.

### Evaluator Weights

| Component | Weight | Description |
|-----------|--------|-------------|
| Filler removal | 25% | Remove "make sure to", "ensure that", etc. |
| Conciseness | 20% | Target 40-70% reduction (penalize over-reduction) |
| **Code preservation** | **25%** | Keep code blocks, consolidate variations |
| Structure | 15% | Preserve frontmatter, sections, code |
| Verbose removal | 15% | Remove acronym expansions |

### Expected Output

A well-optimized skill should:
- Remove all filler phrases
- Keep essential kubectl/bash commands
- Consolidate `kubectl get pods -n default/kube-system/my-namespace` into `kubectl get pods [-n <namespace>]`
- Remove unrelated content (PDF, Git commands)
- Achieve 40-70% size reduction while preserving functionality